[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_19_Streaming_Agents_SSE_Realtime_UIs.ipynb)

# 📡 Lesson 19: Streaming Agents — SSE & Real-Time UIs

**Course:** Learn AI → Phase 3: Production AI Engineering  
**Date:** 2026-05-18  
**Prerequisite:** Lesson 18 (AI Security — Prompt Injection, Guardrails, Red-Teaming)

---

## 🎯 What You'll Learn

By the end of this lesson you will be able to:
1. **Stream LLM responses** token-by-token using the Anthropic Python SDK
2. Read and emit the **Server-Sent Events (SSE)** wire format — the protocol behind every ChatGPT-like UI you've ever seen
3. Stream **complex agents** (ReAct: thoughts → tool calls → results → final answer) using **typed event channels**, so a frontend can render each step distinctly
4. Build a live **Gradio chat UI** that paints tokens as they arrive
5. Reconcile streaming with the **guardrails** you built in Lesson 18 — what changes when you can't see the full reply before showing it

> 🧠 **Why now?** Lesson 16 made your agent reachable on the internet. Lesson 17 measured its quality. Lesson 18 made it safe. Lesson 19 makes it **feel fast** — perceived latency is the #1 differentiator between a demo and a product.


---
## 🧠 Concept: Why Streaming Matters

Without streaming:
```
user asks ──▶ ████████████████████ wait 8s ████████████████████ ──▶ entire reply appears
```

With streaming:
```
user asks ──▶ █ 200ms ▶ first token ──▶ tokens paint progressively ──▶ done at ~8s
                              (same total cost, totally different feel)
```

The total time is **the same**. The total cost is **the same**. But the user *experiences* the agent as ~40× faster because something is happening from 200ms onward.

### Three places streaming earns its keep

| Use case | What you stream | Why it helps |
|----------|-----------------|--------------|
| **Chat UI** | Final-answer tokens | "Is this thing alive?" → resolved in 200ms |
| **Agent observability** | ReAct trace (`thinking`, `tool_call`, `tool_result`) | The user sees *why* the agent is taking 30s — building trust |
| **Early abort** | Cancellable HTTP request | User clicks Stop → save the rest of the tokens (real $ savings on long generations) |

### Cost & quality are unchanged

Streaming does **not** change:
- The number of input/output tokens billed
- The model's output quality
- The total wall-clock latency

It only changes **what the user sees while waiting**.

### What streaming *does* change

It **breaks any pattern that assumes you have the full response before showing it.** That includes:
- **Output guardrails** (you'd have to wait to classify the full text → defeats streaming)
- **Structured-output validation** (you can't `Pydantic.parse_raw` half a JSON)
- **Logging** (you need to assemble the full reply at the end for the audit log)

We'll address all three later in the lesson.


---
## ⚙️ Setup — Install Packages & Load API Key


In [ ]:
# Streaming + UI + server tooling
!pip install anthropic gradio fastapi uvicorn nest_asyncio sse-starlette rich -q
print("✅ Packages installed")


In [ ]:
import os, json, asyncio, time, re
from typing import Iterator, Generator, Callable

# Load API key from Colab Secrets (Runtime → Secrets → ANTHROPIC_API_KEY)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    os.environ.setdefault("ANTHROPIC_API_KEY", "your-api-key-here")
    print("⚠️  Using fallback API key — set ANTHROPIC_API_KEY locally if needed")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap — perfect for streaming demos
print("✅ Anthropic client ready, model:", MODEL)


---
# 🟢 Part 1: Anthropic Streaming Primitives

The Anthropic Python SDK exposes streaming through a **context manager** that yields a `MessageStream` object:

```python
with client.messages.stream(model=..., messages=...) as stream:
    for chunk in stream.text_stream:     # plain text deltas
        print(chunk, end="", flush=True)
    final = stream.get_final_message()   # full Message after stream ends
```

Three levels of detail, from easy → powerful:

| API surface | What you get | When to use |
|-------------|--------------|-------------|
| `stream.text_stream` | Text chunks only (str) | 90% of cases — chat UIs |
| `for event in stream:` | Typed events (deltas, block starts/stops, usage) | Tool use, multi-block, custom UIs |
| `client.messages.create(..., stream=True)` | Raw event iterator (no helpers) | Building your own SDK / proxying |


### 1.1 — The simplest possible stream


In [ ]:
print("Claude is thinking", end="", flush=True)
t0 = time.time()
first_token_at = None

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "In 4 short sentences, explain why streaming improves perceived UX for chat apps."}],
) as stream:
    print("\n\n", end="")
    for text in stream.text_stream:
        if first_token_at is None:
            first_token_at = time.time()
        print(text, end="", flush=True)
    final = stream.get_final_message()

print(f"\n\n⏱  first token: {(first_token_at - t0)*1000:.0f} ms"
      f"  |  total: {(time.time() - t0)*1000:.0f} ms"
      f"  |  output tokens: {final.usage.output_tokens}")

# 💡 EXPERIMENT: ask for a 2000-word essay and watch the gap between
# "first token" and "total" balloon. That's exactly the gap streaming hides.


### 1.2 — The full event stream

Under the hood, the SDK is consuming a sequence of typed events. Knowing them by name is essential the moment you go beyond plain text — tool use, multiple content blocks, image generation, etc.

| Event | Meaning |
|-------|---------|
| `message_start` | New response is starting; carries the empty `Message` shell + model name + initial usage |
| `content_block_start` | A new block is starting. Type is `text` or `tool_use` |
| `content_block_delta` | A delta on the current block: `text_delta` adds text, `input_json_delta` adds JSON to a tool call's `input` field |
| `content_block_stop` | The current block is complete |
| `message_delta` | Top-level metadata update — usually `stop_reason` and final `usage` |
| `message_stop` | End of the message. No more events. |


In [ ]:
# Same call as before, but iterate raw typed events so we can see the protocol.
from collections import Counter
event_counts = Counter()

with client.messages.stream(
    model=MODEL,
    max_tokens=120,
    messages=[{"role": "user", "content": "Say hello in exactly 2 sentences."}],
) as stream:
    for ev in stream:
        event_counts[ev.type] += 1
        if ev.type == "content_block_delta" and ev.delta.type == "text_delta":
            # this is where the visible characters arrive
            pass
        elif ev.type == "message_delta":
            print(f"[message_delta] stop_reason={ev.delta.stop_reason}  usage_out={ev.usage.output_tokens}")
        elif ev.type in {"message_start", "content_block_start", "content_block_stop", "message_stop"}:
            print(f"[{ev.type}]")

print("\nEvent counts:", dict(event_counts))

# 💡 EXPERIMENT: replace text_delta handling with `print(ev.delta.text, end='')`
# to see the same painting behavior as Part 1.1, but built from raw events.


### 1.3 — Streaming **with tools**

When the model decides to call a tool mid-stream, you'll see:
1. `content_block_start` with `content_block.type == "tool_use"` (you now know the tool name and its `id`)
2. A series of `content_block_delta` events with `delta.type == "input_json_delta"` — each one carries a **fragment of the JSON arguments**
3. `content_block_stop` — the JSON is now complete; you can parse it

You typically *don't* execute the tool until `content_block_stop` (otherwise you'd run on partial args). But you can use `input_json_delta` to *render* "Claude is calling `web_search` with `query='...'`" progressively to the user.


In [ ]:
# A tiny synthetic tool just to show the streaming shape
TOOLS = [{
    "name": "get_weather",
    "description": "Return weather for a city.",
    "input_schema": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
}]

with client.messages.stream(
    model=MODEL,
    max_tokens=400,
    tools=TOOLS,
    messages=[{"role": "user", "content": "What's the weather in Tokyo and Paris?"}],
) as stream:
    tool_buf = ""
    current_tool = None
    for ev in stream:
        if ev.type == "content_block_start":
            blk = ev.content_block
            if blk.type == "tool_use":
                current_tool = blk.name
                tool_buf = ""
                print(f"\n🛠  Claude wants to call tool: {blk.name}  (id={blk.id})")
            elif blk.type == "text":
                print("\n💬 text block starting")
        elif ev.type == "content_block_delta":
            if ev.delta.type == "text_delta":
                print(ev.delta.text, end="", flush=True)
            elif ev.delta.type == "input_json_delta":
                tool_buf += ev.delta.partial_json
                print(f"   ⏳ partial args: {tool_buf!r}")
        elif ev.type == "content_block_stop":
            if current_tool:
                print(f"   ✅ tool args complete: {tool_buf}")
                current_tool = None

    final = stream.get_final_message()

print("\nstop_reason:", final.stop_reason)
print("blocks:", [b.type for b in final.content])

# 💡 EXPERIMENT: the tool will NOT actually be executed by Claude — that's *your*
# job (review Lesson 3). All the SDK does is stream the model's intent. In Part 4
# we'll close the loop: execute the tool and feed its result back.


---
# 📡 Part 2: Server-Sent Events (SSE)

So far, streaming has happened **inside Python**. But the user is in a browser. How do the tokens get from your FastAPI server to their `<div>`?

The answer is **Server-Sent Events (SSE)** — a vanilla HTTP feature where the server keeps a single long-lived response open and pushes text frames.

### The wire format

An SSE response is just a normal HTTP response with `Content-Type: text/event-stream`. The body is a series of frames separated by **blank lines**:

```
event: token
data: {"text": "Hello"}

event: token
data: {"text": " world"}

event: done
data: {"usage": {"output_tokens": 2}}

```

Each frame is **two lines**:
- `event: <name>`   ← optional; defaults to `"message"`
- `data: <payload>` ← the payload (often JSON)

Frame ends with `\n\n`. Browsers consume this with one line of JavaScript:

```js
const es = new EventSource("/api/stream");
es.addEventListener("token",  e => append(JSON.parse(e.data).text));
es.addEventListener("done",   e => es.close());
```

### Why SSE and not WebSockets?

| | SSE | WebSocket |
|--|-----|-----------|
| Direction | server → client only | bidirectional |
| Protocol | plain HTTP | upgrade handshake |
| Reconnect | automatic, built-in | you handle it |
| Works through CDNs | ✅ | sometimes 😬 |
| Right for chat token streams | ✅ | overkill |
| Right for collaborative editing | — | ✅ |

For LLM token streaming, SSE is the boring-and-correct choice.


### 2.1 — Hand-build the SSE format from a Claude stream

Let's write a function that converts an Anthropic stream into properly-framed SSE bytes. This is *exactly* what a FastAPI route would yield.


In [ ]:
def claude_to_sse(user_msg: str) -> Iterator[str]:
    """Yield SSE-framed strings. Each yield is one frame (ends in '\n\n')."""
    def frame(event: str, data: dict) -> str:
        return f"event: {event}\ndata: {json.dumps(data)}\n\n"

    yield frame("start", {"model": MODEL})

    with client.messages.stream(
        model=MODEL,
        max_tokens=200,
        messages=[{"role": "user", "content": user_msg}],
    ) as stream:
        for text in stream.text_stream:
            yield frame("token", {"text": text})
        final = stream.get_final_message()

    yield frame("done", {"usage": {
        "input":  final.usage.input_tokens,
        "output": final.usage.output_tokens,
    }})


# Print what a browser would receive over the wire.
print("─── WIRE BYTES (what curl would see) ─────────────────────────────")
for line in claude_to_sse("Give me a one-sentence haiku about streaming."):
    print(line, end="")
print("─" * 64)

# 💡 EXPERIMENT: pipe each yielded frame through a parser. The Python
# equivalent of the browser EventSource:
#
#     for line in stream.iter_lines():
#         if line.startswith('data: '):
#             payload = json.loads(line[6:])
#             render(payload)


### 2.2 — Wrap it in a FastAPI endpoint

This is the production shape. Drop into `main.py`, deploy with the Dockerfile from Lesson 16, and you're streaming to real browsers.

```python
# main.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.get("/api/stream")
def stream(prompt: str):
    return StreamingResponse(
        claude_to_sse(prompt),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",    # disable nginx buffering
            "Connection": "keep-alive",
        },
    )
```

That's it. The whole HTTP-streaming story is **`text/event-stream` + a generator that yields**.

> ⚠️  **Gotcha #1 — Buffering.** If your reverse proxy (nginx, Cloudflare, some load balancers) buffers responses, your SSE stream will arrive all at once at the browser. Fix: `X-Accel-Buffering: no` (nginx) or use HTTP/2 + a proxy that respects `Content-Type: text/event-stream`.
>
> ⚠️  **Gotcha #2 — Browser timeout.** Some clients drop idle connections after ~30s of silence. If a long agent run goes quiet, emit a heartbeat: `yield "event: ping\ndata: {}\n\n"` every 15s.
>
> ⚠️  **Gotcha #3 — CORS.** SSE responses are subject to the same CORS rules as fetches. Set `Access-Control-Allow-Origin` on your FastAPI app.


In [ ]:
# Simulate the browser side: parse SSE bytes back into events.
def parse_sse(stream_bytes: Iterator[str]):
    buffer = ""
    for chunk in stream_bytes:
        buffer += chunk
        while "\n\n" in buffer:
            frame, buffer = buffer.split("\n\n", 1)
            ev, data = {}, {}
            for line in frame.split("\n"):
                if line.startswith("event: "):
                    ev["event"] = line[7:]
                elif line.startswith("data: "):
                    data = json.loads(line[6:])
            yield ev.get("event", "message"), data


print("🌐 Browser-side render:\n")
output = ""
for event_name, payload in parse_sse(claude_to_sse("Write a haiku about JSON.")):
    if event_name == "start":
        print(f"[stream opened from {payload['model']}]")
    elif event_name == "token":
        output += payload["text"]
        print(payload["text"], end="", flush=True)
    elif event_name == "done":
        print(f"\n\n[done — tokens: {payload['usage']}]")


---
# 🤖 Part 3: Typed Event Channels for Agent Streaming

Plain text streaming is fine for a one-shot Q&A. For an **agent** — which thinks, calls tools, sees results, and only then answers — plain text is the *wrong* abstraction.

Agents produce **structured intermediate state**:

```
🧠  thought         → the model's free-text reasoning
🛠  tool_call       → tool name + args
📦  tool_result     → what the tool returned
✅  final_answer    → the user-facing reply
```

A good streaming agent emits each of these as a **distinct event type** so the frontend can render them differently — thoughts in italic grey, tool calls in collapsible boxes, results in a code block, answer in the normal bubble.

This is exactly what the ChatGPT, Claude.ai, and Cursor UIs do behind the scenes.


### 3.1 — Define the event types

We'll use a simple dict schema. In production you'd type this with Pydantic (Lesson 10) and validate at the FastAPI layer.


In [ ]:
# Event schema
# Every event is a dict: {"type": str, ...payload}

EVENT_TYPES = {
    "thought":       "Model's reasoning text — append to a thoughts panel",
    "tool_call":     "Model is calling a tool (name + input)",
    "tool_result":   "Result of a tool call (executed by our code)",
    "final_answer":  "User-facing answer — append to the chat bubble",
    "error":         "Something went wrong; close the stream",
    "done":          "End of run; usage stats included",
}

for t, desc in EVENT_TYPES.items():
    print(f"  {t:14s} → {desc}")


### 3.2 — A tiny tool registry

We need real tools to demonstrate. Two pretend tools — no network, deterministic output.


In [ ]:
def tool_web_search(query: str) -> str:
    # Deterministic fake — pretend Wikipedia
    return f"[search] Top hit for '{query}': SSE is a unidirectional HTTP-based push protocol introduced in HTML5."

def tool_calculator(expression: str) -> str:
    try:
        return f"[calc] {expression} = {eval(expression, {'__builtins__': {}}, {})}"
    except Exception as e:
        return f"[calc error] {e}"

TOOL_FUNCS = {
    "web_search": tool_web_search,
    "calculator": tool_calculator,
}

TOOL_SCHEMAS = [
    {"name": "web_search", "description": "Search the web for a short snippet.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "calculator", "description": "Evaluate a Python arithmetic expression.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
]
print("Registry ready:", list(TOOL_FUNCS))


### 3.3 — The streaming ReAct agent

This is the heart of the lesson. Read it carefully.

The function is a **generator** that yields events. The caller can iterate it once and:
- Print events to the console (our demo)
- Forward them as SSE frames (the FastAPI version)
- Feed them into Gradio's chat panel (Section 4)

Same agent, three different sinks. That's the power of typed event streams.


In [ ]:
def streaming_react_agent(user_msg: str, max_iters: int = 4) -> Generator[dict, None, None]:
    """
    Yields events of the EVENT_TYPES schema.

    Loop:  stream a model turn → emit thoughts → on tool_use blocks, execute and
           append the result as a user message → next turn → ...
    Terminates when stop_reason == 'end_turn' or after `max_iters` rounds.
    """
    messages = [{"role": "user", "content": user_msg}]
    total_usage = {"input": 0, "output": 0}

    for iteration in range(max_iters):
        # ---- Stream one turn from the model ----
        with client.messages.stream(
            model=MODEL,
            max_tokens=1024,
            tools=TOOL_SCHEMAS,
            messages=messages,
        ) as stream:

            tool_json_buf = ""
            current_tool_block = None  # (id, name)

            for ev in stream:
                if ev.type == "content_block_start":
                    blk = ev.content_block
                    if blk.type == "tool_use":
                        current_tool_block = (blk.id, blk.name)
                        tool_json_buf = ""

                elif ev.type == "content_block_delta":
                    if ev.delta.type == "text_delta":
                        # Text deltas during a turn that ends with end_turn are the
                        # final answer; otherwise they're 'thoughts' that precede a tool call.
                        # We can't know yet, so we tentatively emit as 'thought' and
                        # promote to 'final_answer' at end_turn (see below).
                        yield {"type": "thought", "delta": ev.delta.text}
                    elif ev.delta.type == "input_json_delta":
                        tool_json_buf += ev.delta.partial_json

                elif ev.type == "content_block_stop":
                    if current_tool_block is not None:
                        tool_id, tool_name = current_tool_block
                        try:
                            args = json.loads(tool_json_buf)
                        except json.JSONDecodeError:
                            args = {}
                        yield {"type": "tool_call", "id": tool_id, "name": tool_name, "input": args}

                        # Execute the tool
                        if tool_name in TOOL_FUNCS:
                            result = TOOL_FUNCS[tool_name](**args)
                        else:
                            result = f"[error] unknown tool {tool_name}"
                        yield {"type": "tool_result", "id": tool_id, "output": result}
                        current_tool_block = None

            final = stream.get_final_message()
            total_usage["input"]  += final.usage.input_tokens
            total_usage["output"] += final.usage.output_tokens

        # ---- Decide whether to loop or stop ----
        if final.stop_reason == "end_turn":
            # Text we already streamed was the final answer — promote.
            text_blocks = [b.text for b in final.content if b.type == "text"]
            yield {"type": "final_answer", "text": "".join(text_blocks)}
            break

        if final.stop_reason == "tool_use":
            # Append assistant turn + tool_result user turn, then loop.
            messages.append({"role": "assistant", "content": final.content})
            tool_results = []
            for b in final.content:
                if b.type == "tool_use":
                    fn = TOOL_FUNCS.get(b.name, lambda **kw: f"[unknown tool {b.name}]")
                    out = fn(**b.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": b.id,
                        "content": out,
                    })
            messages.append({"role": "user", "content": tool_results})
            continue

        # Any other stop_reason → emit and exit
        yield {"type": "error", "reason": f"unexpected stop_reason={final.stop_reason}"}
        break

    yield {"type": "done", "usage": total_usage}


### 3.4 — Run the streaming agent

Now we'll use it: connect to one of the toy tools, watch every event print in real time, and inspect the final state.


In [ ]:
ICON = {"thought": "🧠", "tool_call": "🛠 ", "tool_result": "📦",
        "final_answer": "✅", "error": "💥", "done": "🏁"}

# Track total state for display
final_answer = ""
print("─── streaming agent ─────────────────────────────────────────\n")
for event in streaming_react_agent(
    "What is 173 × 22, and one fact about SSE? Use the tools."
):
    t = event["type"]
    if t == "thought":
        print(event["delta"], end="", flush=True)
    elif t == "tool_call":
        print(f"\n{ICON[t]} call → {event['name']}({event['input']})")
    elif t == "tool_result":
        print(f"{ICON[t]} result → {event['output']}")
    elif t == "final_answer":
        final_answer = event["text"]
        # The text was already streamed as 'thought' deltas above — this is the
        # canonical aggregated answer for the audit log.
    elif t == "done":
        print(f"\n\n{ICON[t]} usage: {event['usage']}")
    elif t == "error":
        print(f"\n{ICON[t]} {event['reason']}")

print("\n📝 Final answer (assembled):", repr(final_answer)[:200])

# 💡 EXPERIMENT: ask a question that needs only the calculator, or only search,
# or neither — the agent should issue the right number of tool calls (0, 1, or
# more) and the event stream changes shape accordingly."""


> 🧠 **Note on thoughts vs. final answer.** Claude doesn't expose a separate "reasoning channel" in its public API the way some other models do. In our stream above, all `text_delta`s are emitted as `thought` deltas; we promote the **last** turn's text to `final_answer` because that's the turn where `stop_reason == "end_turn"`. A more sophisticated frontend would render only the final turn's text in the chat bubble and show earlier turns' text in a collapsible "thoughts" panel.


---
# 🛡  Part 4: Streaming Meets Lesson 18's Guardrails

Recall the four-layer defense from Lesson 18:

```
user → [L3 input guardrail] → [L1 hardened LLM] → [L4 output guardrail] → user
```

Streaming **breaks** L4. The whole point of L4 was *don't send the reply to the user if it leaks secrets* — but if we're streaming tokens directly to the user, by the time we've classified the full reply, the user has already seen it.

There are three honest options:

| Strategy | Streaming UX | Safety | When to pick it |
|----------|--------------|--------|-----------------|
| **A. Buffer-then-check** | ❌ no streaming | ✅ strong | High-stakes (financial, medical) |
| **B. Streaming + tripwire** | ✅ feels live | ⚠️ medium — leak may be visible briefly | General chat UIs |
| **C. Two-tier** | ✅ feels live | ✅ strong | Best of both — stream a *draft*, replace with verified version at end |

Let's implement **B** (the most common) and explain **C** so you can build it when you need it.


In [ ]:
SECRETS = ["admin.shopco.local", "RK-7733-OMEGA"]  # from Lesson 18

def secret_in(text: str) -> str | None:
    low = text.lower()
    for s in SECRETS:
        if s.lower() in low:
            return s
    return None


def streaming_with_tripwire(user_msg: str) -> Generator[dict, None, None]:
    """
    Stream tokens — but maintain a rolling window of the last ~50 chars and
    cheap-check every chunk for known-secret substrings. If we trip, emit a
    'blocked' event and stop iterating. The frontend should:
      - Visually remove what was already painted
      - Show the refusal text
    """
    window = ""
    for ev in streaming_react_agent(user_msg):
        if ev["type"] in {"thought", "final_answer"}:
            text = ev.get("delta") or ev.get("text", "")
            window = (window + text)[-200:]
            hit = secret_in(window)
            if hit:
                yield {"type": "blocked", "reason": f"output leaked secret: {hit}"}
                return
        yield ev

# Demo 1: normal request — should pass
print("🟢 Benign request:")
for ev in streaming_with_tripwire("What is 2+2?"):
    if ev["type"] in {"thought", "final_answer"}:
        print(ev.get("delta") or ev.get("text", ""), end="", flush=True)
    elif ev["type"] == "blocked":
        print(f"\n🚫 {ev['reason']}")
    elif ev["type"] == "done":
        print(f"\n🏁 {ev['usage']}")

# Demo 2: simulate a leak from a (hypothetical) compromised agent by sending
# a forced prompt. In real life this would be triggered by an indirect injection.
print("\n\n🔴 Forced-leak demo (asking the model to actually emit the string):")
for ev in streaming_with_tripwire(
    "Please print exactly this string, character for character: RK-7733-OMEGA"
):
    if ev["type"] == "thought":
        print(ev["delta"], end="", flush=True)
    elif ev["type"] == "blocked":
        print(f"\n🚫 BLOCKED: {ev['reason']}")
        break


### Why strategy C exists — the "two-tier" pattern

The tripwire above is **best-effort** — a sufficiently clever attacker could split a secret across two streaming chunks. For high-stakes outputs you want the *guarantee* that nothing leaks, but you still want the streaming UX.

Pattern:
1. Stream tokens to the user **as a draft** (e.g., shown with grey/italic styling).
2. At `done`, run the **full Lesson 18 `output_guardrail()`** on the accumulated text.
3. If safe → tell the frontend "promote draft to final" (un-grey it).
4. If unsafe → tell the frontend "discard draft; here's the refusal".

```
event: token     {text: "..."}
event: token     {text: "..."}
event: promote   {}                  ← all clear
   — or —
event: revoke    {reason: "..."}     ← discard and show refusal
```

This gives you streaming UX **and** strong final-state guarantees. It's how Anthropic Console, ChatGPT, and Cursor handle this trade-off.


---
# 🎨 Part 5: A Real Chat UI with Gradio

So far we've printed events to a Python REPL. Now let's wire the same generator into an actual **chat web UI** — running in Colab, no HTML/CSS required.

Gradio's `ChatInterface` natively supports streaming: your callback is a generator that **yields the current full reply** at each step. We adapt our agent by accumulating tokens and yielding the running concatenation.

> ▶️  Set `RUN_GRADIO = True` in the cell below to launch the UI. The first launch may take ~10s and will print a public `share` URL you can click.


In [ ]:
RUN_GRADIO = False   # flip to True to launch the web UI

if RUN_GRADIO:
    import gradio as gr

    def chat_fn(user_msg, history):
        """Gradio expects yielded strings (the running full reply)."""
        running = ""
        thoughts_log = []
        for ev in streaming_with_tripwire(user_msg):
            t = ev["type"]
            if t == "thought":
                running += ev["delta"]
                yield running
            elif t == "tool_call":
                running += f"\n\n> 🛠  *Calling `{ev['name']}({ev['input']})`*\n\n"
                yield running
            elif t == "tool_result":
                running += f"> 📦 *Result:* `{ev['output']}`\n\n"
                yield running
            elif t == "blocked":
                yield f"🚫 Reply blocked: {ev['reason']}"
                return
            elif t == "done":
                running += f"\n\n*— tokens: {ev['usage']} —*"
                yield running

    demo = gr.ChatInterface(
        fn=chat_fn,
        type="messages",
        title="📡 Streaming Agent — Lesson 19",
        description="Ask anything. Watch the agent think, call tools, and answer in real time.",
        examples=["What is 5 factorial?", "Tell me one fact about SSE and one about HTTP/2."],
    )
    demo.launch(share=True, debug=False)
else:
    print("Gradio is disabled. Flip RUN_GRADIO = True above to launch the UI.")
    print("Why disabled by default? It opens a long-lived server that would "
          "block the rest of the notebook from running top-to-bottom.")

# 💡 EXPERIMENT: once the UI is running, paste in an indirect-injection from
# Lesson 18's ATTACK_CATALOG. See the tripwire fire mid-stream.


---
# 🏆 Part 6: Capstone — Streaming AutoResearcher Endpoint

Time to wire everything together into the **exact code** you'd ship in `autoresearcher/streaming.py` and import from `main.py`.

The capstone has three layers:
1. **Agent core** — `streaming_react_agent()` from Part 3
2. **Safety wrapper** — `streaming_with_tripwire()` from Part 4
3. **SSE transport** — converts events to wire frames; ready to plug into FastAPI


In [ ]:
def agent_to_sse(user_msg: str) -> Iterator[str]:
    """Production-shape: bytes-over-HTTP for FastAPI StreamingResponse."""
    def frame(event: str, data: dict) -> str:
        return f"event: {event}\ndata: {json.dumps(data)}\n\n"

    yield frame("start", {"model": MODEL})
    try:
        for ev in streaming_with_tripwire(user_msg):
            t = ev["type"]
            # 1:1 mapping — SSE event name == agent event type
            payload = {k: v for k, v in ev.items() if k != "type"}
            yield frame(t, payload)
    except Exception as e:
        yield frame("error", {"reason": str(e)})

# Smoke test — print the actual wire bytes the browser would consume
print("─── WIRE BYTES (first 1.5 KB) ─────────────────────────────────\n")
collected = ""
for f in agent_to_sse("What's 12*12?"):
    collected += f
    if len(collected) > 1500:
        break
print(collected[:1500])
print("… [truncated]")


### How to ship this

In your `autoresearcher` repo, replace your existing non-streaming endpoint:

```python
# autoresearcher/server.py
from fastapi import FastAPI, Depends, Header, HTTPException
from fastapi.responses import StreamingResponse
from .streaming import agent_to_sse        # ← the function above

app = FastAPI()

def auth(x_api_key: str = Header(...)):
    if x_api_key != SETTINGS.api_key:
        raise HTTPException(401, "bad key")

@app.get("/api/agent/stream")
def stream(prompt: str, _=Depends(auth)):
    return StreamingResponse(
        agent_to_sse(prompt),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )
```

Combined with Lesson 16 (Docker + fly.toml), Lesson 17 (eval gate in CI), and Lesson 18 (`SecureAgent` wrapper around the tool dispatcher), you now have:

✅ deployed → ✅ evaluated → ✅ secured → ✅ **streamed**

### Browser-side reference (HTML + 20 lines of JS)

```html
<div id="chat"></div>
<script>
  const es = new EventSource('/api/agent/stream?prompt=Hello');
  const chat = document.getElementById('chat');
  let answer = document.createElement('div');
  chat.appendChild(answer);

  es.addEventListener('thought', e => {
    answer.textContent += JSON.parse(e.data).delta;
  });
  es.addEventListener('tool_call', e => {
    const d = JSON.parse(e.data);
    chat.appendChild(Object.assign(document.createElement('div'),
      {className: 'tool', textContent: `🛠 ${d.name}(${JSON.stringify(d.input)})`}));
  });
  es.addEventListener('blocked', e => {
    answer.textContent = '🚫 ' + JSON.parse(e.data).reason;
    es.close();
  });
  es.addEventListener('done', () => es.close());
</script>
```


---
# 📚 Lesson 19 Summary

## What you built today

| Component | Role |
|-----------|------|
| `client.messages.stream(...)` | Anthropic SDK primitive — text + raw events |
| `claude_to_sse` | Convert Claude's stream into wire-format SSE frames |
| `parse_sse` | Browser-side counterpart — reconstruct events from wire bytes |
| `streaming_react_agent` | Typed-event generator (thought / tool_call / tool_result / final_answer / done) |
| `streaming_with_tripwire` | Lesson 18 guardrails, streaming-friendly |
| Gradio `ChatInterface` | Live chat UI with token-by-token painting |
| `agent_to_sse` | Production: agent events → SSE frames → FastAPI StreamingResponse |

## Mental models to keep

1. **Streaming changes UX, not cost.** Same tokens, same dollars — but the user perceives the agent as ~40× faster.
2. **SSE is the boring-correct protocol.** Plain HTTP, one direction, automatic reconnect. WebSockets are overkill for chat.
3. **Typed events > plain text.** Once an agent has more than one phase (thoughts, tools, answer), give each phase its own event type. Your frontend will thank you.
4. **Streaming breaks output guardrails.** You can do a tripwire (cheap, leaky) or a two-tier draft/promote pattern (strong, slightly more UI work). High-stakes domains should buffer.
5. **The agent is a generator.** Same generator → console demo → Gradio UI → SSE endpoint → CLI. One implementation, four sinks.

## Beyond this notebook

- **Backpressure & cancellation.** Real clients close the connection when the user hits Stop. Wrap your agent loop in a check for `request.is_disconnected()` (FastAPI) and break early.
- **Persistence.** Stream the events to the user *and* `append` them to a database row so you can replay the run later (great for debugging + audit logs).
- **Multi-user fairness.** A single agent stream can hog a worker for 30s. Use async (`AsyncAnthropic`) so one process can serve many concurrent streams. (We'll touch this in Lesson 22 — Cost Engineering.)

## What's next

**Lesson 20 → Vector DB in Production** — graduating from in-memory ChromaDB (Lesson 7) to Pinecone / Weaviate / pgvector. Indexing strategies (HNSW, IVF), hybrid search (vector + keyword), metadata filtering at scale, and how to keep latency under 100ms when your index has 50M vectors.

---
*Lesson 19 of 23 — Phase 3: Production AI Engineering*
